In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table("fintech360.bronze.bureau_data")
display(df)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    DecimalType, BooleanType, LongType, IntegerType
)

BUREAU_PARENT_SCHEMA = StructType([
    StructField("bureau_record_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("bureau_provider", StringType(), True),
    StructField("report_ts", TimestampType(), True),
    StructField("credit_score", IntegerType(), True),
    StructField("active_accounts", LongType(), True),
    StructField("total_accounts", LongType(), True),
    StructField("total_outstanding_amount", DecimalType(18, 2), True),
    StructField("overdue_amount", DecimalType(18, 2), True),
    StructField("enquiries_last_30d", LongType(), True),
    StructField("enquiries_last_90d", LongType(), True),
    StructField("delinquency_ever", BooleanType(), True),
    StructField("written_off_accounts", LongType(), True),
    StructField("report_reference", StringType(), True),
    StructField("created_at", LongType(), True),
    StructField("updated_at", LongType(), True),
    StructField("bureau_details_json", StringType(), True),
    StructField("_rescued_data", StringType(), True),
    StructField("_ingested_at", TimestampType(), True),
    StructField("_source_file", StringType(), True)
])

In [0]:
from pyspark.sql import functions as F

def normalize_bureau_schema(df, parent_schema=BUREAU_PARENT_SCHEMA):
    parent_names = [field.name for field in parent_schema.fields]
    parent_fields = {field.name: field for field in parent_schema.fields}

    # Identify unexpected source columns
    extra_cols = [c for c in df.columns if c not in parent_names]

    # Preserve unexpected columns in the rescue payload
    if extra_cols:
        extra_json = F.to_json(
            F.struct(*[F.col(c) for c in extra_cols])
        )

        if "_rescued_data" in df.columns:
            df = df.withColumn(
                "_rescued_data",
                F.when(
                    F.col("_rescued_data").isNull(),
                    extra_json
                ).otherwise(
                    F.concat_ws(
                        "\n",
                        F.col("_rescued_data"),
                        extra_json
                    )
                )
            )
        else:
            df = df.withColumn("_rescued_data", extra_json)

    # Normalize IDs without retaining unnecessary ".0" suffixes
    for col_name in ["bureau_record_id", "customer_id"]:
        if col_name in df.columns:
            value = F.trim(F.col(col_name).cast("string"))
            numeric_value = value.cast("decimal(38,10)")

            df = df.withColumn(
                col_name,
                F.when(
                    value.rlike(r"^[+-]?\d+\.0+$"),
                    numeric_value.cast("decimal(38,0)").cast("string")
                ).otherwise(value)
            )

    # Add missing columns and cast existing columns
    for name, field in parent_fields.items():
        if name not in df.columns:
            df = df.withColumn(name, F.lit(None).cast(field.dataType))
        else:
            df = df.withColumn(name, F.col(name).cast(field.dataType))

    # Enforce the parent column order
    return df.select(*parent_names)

In [0]:
df = normalize_bureau_schema(df)

df.printSchema()
display(df.limit(10))

In [0]:
(df.write
    .mode('append')
    .format('delta')
    .saveAsTable('fintech360.schema_validation.bureau_data'))